In [1]:
from pyspark.sql import SparkSession
# lancer l'ui de mlflow
#cd /home/jovyan/work
#mlflow ui --backend-store-uri sqlite:///mlflow.db --host 0.0.0.0 --port 4044


spark = SparkSession.builder \
    .appName("TP3-WineQuality") \
    .getOrCreate()

df = spark.read.csv("winequality-red.csv", header=True, inferSchema=True, sep=",")
df.show(5)
df.printSchema()

+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|fixed acidity|volatile acidity|citric acid|residual sugar|chlorides|free sulfur dioxide|total sulfur dioxide|density|  pH|sulphates|alcohol|quality|
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+
|          7.4|             0.7|        0.0|           1.9|    0.076|               11.0|                34.0| 0.9978|3.51|     0.56|    9.4|      5|
|          7.8|            0.88|        0.0|           2.6|    0.098|               25.0|                67.0| 0.9968| 3.2|     0.68|    9.8|      5|
|          7.8|            0.76|       0.04|           2.3|    0.092|               15.0|                54.0|  0.997|3.26|     0.65|    9.8|      5|
|         11.2|            0.28|       0.56|           1.9|    0.075|               17.0|           

In [2]:
from pyspark.ml.feature import VectorAssembler

# Renommer quality en label 
df = df.withColumnRenamed("quality", "label")

# Toutes les colonnes sauf label sont des features
feature_cols = [c for c in df.columns if c != "label"]
print(feature_cols)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(df).select("features", "label")

data.show(5, truncate=False)

['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']
+--------------------------------------------------------+-----+
|features                                                |label|
+--------------------------------------------------------+-----+
|[7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4]  |5    |
|[7.8,0.88,0.0,2.6,0.098,25.0,67.0,0.9968,3.2,0.68,9.8]  |5    |
|[7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.997,3.26,0.65,9.8] |5    |
|[11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.998,3.16,0.58,9.8]|6    |
|[7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4]  |5    |
+--------------------------------------------------------+-----+
only showing top 5 rows



In [3]:
#separer le data set 
train_data, test_data = data.randomSplit([0.7, 0.3], seed=42)

print(f"Train: {train_data.count()} lignes")
print(f"Test: {test_data.count()} lignes")

Train: 1173 lignes
Test: 426 lignes


In [4]:
import mlflow
import mlflow.spark
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

mlflow.set_tracking_uri("http://127.0.0.1:4044")
mlflow.set_experiment("wine-quality-tp3")

with mlflow.start_run():
    lr = LinearRegression(featuresCol="features", labelCol="label")
    lr_model = lr.fit(train_data)
    predictions = lr_model.transform(test_data)

    evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
    evaluator_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
    rmse = evaluator_rmse.evaluate(predictions)
    r2 = evaluator_r2.evaluate(predictions)

    print(f"RMSE: {rmse}")
    print(f"R2: {r2}")

    mlflow.log_param("maxIter", lr.getMaxIter())
    mlflow.log_param("regParam", lr.getRegParam())
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.spark.log_model(lr_model, "linear-regression-model")

RMSE: 0.6670030973323298
R2: 0.3470856805859932
🏃 View run rebellious-skunk-171 at: http://127.0.0.1:4044/#/experiments/1/runs/bab05754d74d406dacc96408786acd4f
🧪 View experiment at: http://127.0.0.1:4044/#/experiments/1


In [5]:
    # Coefficients et intercept
    print("Coefficients:", lr_model.coefficients)
    print("Intercept:", lr_model.intercept)

    # Associer chaque coefficient à son nom de feature
    for name, coef in zip(feature_cols, lr_model.coefficients):
        print(f"{name}: {coef:.4f}")

Coefficients: [0.024504546923137315,-1.097105427261691,-0.24332033090558414,-0.018968324046269917,-1.3762852773410217,0.006309240953362335,-0.0031866947688895127,-0.28200889700073484,-0.4397618939184932,0.8192129389619762,0.3058895297878435]
Intercept: 4.296477856903301
fixed acidity: 0.0245
volatile acidity: -1.0971
citric acid: -0.2433
residual sugar: -0.0190
chlorides: -1.3763
free sulfur dioxide: 0.0063
total sulfur dioxide: -0.0032
density: -0.2820
pH: -0.4398
sulphates: 0.8192
alcohol: 0.3059


In [6]:
# application d'hyper param
for reg_param in [0.0, 0.01, 0.1, 0.5]:
    with mlflow.start_run(run_name=f"regParam={reg_param}"):
        lr = LinearRegression(featuresCol="features", labelCol="label", regParam=reg_param)
        lr_model = lr.fit(train_data)
        predictions = lr_model.transform(test_data)

        rmse = evaluator_rmse.evaluate(predictions)
        r2 = evaluator_r2.evaluate(predictions)

        mlflow.log_param("regParam", reg_param)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.spark.log_model(lr_model, "linear-regression-model")

        print(f"regParam={reg_param} -> RMSE={rmse:.4f}, R2={r2:.4f}")

regParam=0.0 -> RMSE=0.6670, R2=0.3471
🏃 View run regParam=0.0 at: http://127.0.0.1:4044/#/experiments/1/runs/08c93a92fdc34c969b9c77f6fa70f2fa
🧪 View experiment at: http://127.0.0.1:4044/#/experiments/1
regParam=0.01 -> RMSE=0.6665, R2=0.3482
🏃 View run regParam=0.01 at: http://127.0.0.1:4044/#/experiments/1/runs/6b65a28d7d9549cdb422223d505228b4
🧪 View experiment at: http://127.0.0.1:4044/#/experiments/1
regParam=0.1 -> RMSE=0.6660, R2=0.3491
🏃 View run regParam=0.1 at: http://127.0.0.1:4044/#/experiments/1/runs/0386cb66c91e48fb8d600cd8c55ee05f
🧪 View experiment at: http://127.0.0.1:4044/#/experiments/1
regParam=0.5 -> RMSE=0.6804, R2=0.3206
🏃 View run regParam=0.5 at: http://127.0.0.1:4044/#/experiments/1/runs/831a7ced12df44bd8a57ad5c398f4b5f
🧪 View experiment at: http://127.0.0.1:4044/#/experiments/1


In [7]:
# exercice 2 du tp sur l'ui de spark 
df_taxi = spark.read.csv("yellow_tripdata_2015-01.csv", header=True, inferSchema=True)

df_taxi.printSchema()
df_taxi.show(5)
print(f"Nombre de lignes : {df_taxi.count()}")

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+------------------+---------------

In [17]:
#Lors du chargement du CSV, Spark UI montre 5 jobs. 
#La lecture du header et l'inférence du schéma se font en deux jobs (13 ms puis 5 s sur 32 tâches), suivis de show(5) (14 ms, 1 tâche) puis de count() qui se découpe en deux jobs (0,4 s sur 32 tâches, puis 14 ms d'agrégation). 
#Le job d'inférence du schéma est de loin le plus long et constitue le principal goulot d'étranglement.

In [9]:
import time

print(f"Partitions initiales : {df_taxi.rdd.getNumPartitions()}")
# j'ai eu des probleme de connectionrefused JVM qui plantait a cause du volume de donné d'ou le sample
df_sample = df_taxi.sample(fraction=0.01, seed=42)  # ~1% des lignes
print(f"Partitions du sample : {df_sample.rdd.getNumPartitions()}")  # doit afficher 32

df_sample.cache()
df_sample.count()  

# Baseline (32 partitions d'origine)
start = time.time()
df_sample.count()
print(f"Baseline : {time.time() - start:.2f}s, partitions : {df_sample.rdd.getNumPartitions()}")

# coalesce : 32 → 4, sans shuffle
df_c = df_sample.coalesce(4)
start = time.time()
df_c.count()
print(f"coalesce(4) : {time.time() - start:.2f}s, partitions : {df_c.rdd.getNumPartitions()}")

# repartition : 32 → 4, avec shuffle complet
df_r = df_sample.repartition(4)
start = time.time()
df_r.count()
print(f"repartition(4) : {time.time() - start:.2f}s, partitions : {df_r.rdd.getNumPartitions()}")

Partitions initiales : 32
Partitions du sample : 32
Baseline : 0.06s, partitions : 32
coalesce(4) : 0.04s, partitions : 4
repartition(4) : 0.10s, partitions : 4


In [10]:
import time
# sans cache
df_filtered = df_taxi.filter(df_taxi["passenger_count"] > 1)

start = time.time()
print(df_filtered.count())
print(f"count 1 (sans cache) : {time.time() - start:.2f}s")

start = time.time()
print(df_filtered.count())
print(f"count 2 (sans cache) : {time.time() - start:.2f}s")

start = time.time()
df_filtered.groupBy("payment_type").count().show()
print(f"groupBy (sans cache) : {time.time() - start:.2f}s")

3748551
count 1 (sans cache) : 1.11s
3748551
count 2 (sans cache) : 0.94s
+------------+-------+
|payment_type|  count|
+------------+-------+
|           1|2240436|
|           3|   5948|
|           4|   2492|
|           2|1499674|
|           5|      1|
+------------+-------+

groupBy (sans cache) : 1.20s


In [12]:
# avec cache
df_filtered.cache()

start = time.time()
print(df_filtered.count())
print(f"count 1 (avec cache, matérialise) : {time.time() - start:.2f}s")

start = time.time()
print(df_filtered.count())
print(f"count 2 (avec cache) : {time.time() - start:.2f}s")

start = time.time()
df_filtered.groupBy("payment_type").count().show()
print(f"groupBy (avec cache) : {time.time() - start:.2f}s")

3748551
count 1 (avec cache, matérialise) : 0.08s
3748551
count 2 (avec cache) : 0.05s
+------------+-------+
|payment_type|  count|
+------------+-------+
|           1|2240436|
|           3|   5948|
|           4|   2492|
|           2|1499674|
|           5|      1|
+------------+-------+

groupBy (avec cache) : 0.14s
